# MCP：标准化 Agent 与外部能力的连接

**MCP 是应用程序与外部能力提供方之间的连接协议。它让工具的定义、发现和调用有一致约定，减少 Agent 客户端为同一套外部系统重复编写适配代码。MCP 不替代 Function Calling——两者在不同层次解决不同问题。**

## 1 结论与学习目标

学完本章，你需要能够解释以下问题：

1. 为什么多个 Agent 客户端对接同一套外部工具需要标准协议。
2. Host、Client、Server、Tool、Resource 和 Prompt 分别是什么，各自负责什么。
3. 一次 MCP 调用从能力发现到结果返回的完整顺序。
4. MCP 与第 4 章的 Function Calling 分别处于什么层次、解决什么问题。
5. 什么场景适合引入 MCP，什么场景直接用本地 Tool Calling 更合理。

先看全貌：

```mermaid
flowchart TD
    H["Agent 应用 / Host"] -->|"创建 Client"| C["MCP Client"]
    C -->|"JSON-RPC"| S["MCP Server"]
    S -->|"暴露"| T1["Tool: read_file"]
    S -->|"暴露"| T2["Tool: search_text"]
    S -->|"暴露"| T3["Tool: repo_status"]
    C -->|"tools/list 发现"| D["可用工具列表"]
    C -->|"tools/call 调用"| R["工具执行结果"]
    T3 --> FS["文件系统 / git"]
    R --> M["大模型"]
    M --> U["最终回答"]
    U --> W["用户"]

    classDef host fill:#ff6b6b,stroke:#c92a2a,color:#fff
classDef client fill:#ffd43b,stroke:#e67700,color:#333
classDef server fill:#51cf66,stroke:#2b8a3e,color:#fff
classDef model fill:#cc5de8,stroke:#862e9c,color:#fff
classDef tool fill:#4dabf7,stroke:#1971c2,color:#fff
classDef tool2 fill:#20c997,stroke:#087f5b,color:#fff
classDef tool3 fill:#ff922b,stroke:#d9480f,color:#fff
classDef user fill:#f783ac,stroke:#c2255c,color:#fff
classDef storage fill:#a9e34b,stroke:#5c940d,color:#333
classDef app fill:#74c0fc,stroke:#1864ab,color:#fff
classDef mcp fill:#63e6be,stroke:#0b7285,color:#fff

    class H host
    class C client
    class S server
    class T1 tool
    class T2 tool2
    class T3 tool3
    class D,R tool2
    class FS storage
    class M model
    class W user
```

## 2 背景与原因：为什么需要 MCP

**MCP 出现的原因是工程摩擦——多个 Agent 客户端为同一套外部能力重复编写适配代码——而不是 Function Calling 本身存在缺陷。**

### 2.1 一个重复适配的场景

假设你维护了一个代码仓库分析工具，包含三个能力：读取文件、搜索文本、获取仓库状态。现在有三个 Agent 客户端需要使用它：

- VS Code Copilot Chat
- Claude Desktop
- 你自建的终端 Agent

在没有标准协议的情况下，每种客户端需要各自对接：

```mermaid
flowchart LR
    C1["Copilot Chat"] -->|"自定义 API"| T["仓库工具"]
    C2["Claude Desktop"] -->|"MCP Server"| T
    C3["终端 Agent"] -->|"直接 import"| T

    classDef host fill:#ff6b6b,stroke:#c92a2a,color:#fff
classDef client fill:#ffd43b,stroke:#e67700,color:#333
classDef server fill:#51cf66,stroke:#2b8a3e,color:#fff
classDef model fill:#cc5de8,stroke:#862e9c,color:#fff
classDef tool fill:#4dabf7,stroke:#1971c2,color:#fff
classDef tool2 fill:#20c997,stroke:#087f5b,color:#fff
classDef tool3 fill:#ff922b,stroke:#d9480f,color:#fff
classDef user fill:#f783ac,stroke:#c2255c,color:#fff
classDef storage fill:#a9e34b,stroke:#5c940d,color:#333
classDef app fill:#74c0fc,stroke:#1864ab,color:#fff
classDef mcp fill:#63e6be,stroke:#0b7285,color:#fff

    class C1 host
    class C2 client
    class C3 model
    class T tool
```

三种不同的对接方式，没有统一协议。这就是 MCP 要解决的问题。

三种对接方式，三种不同的：

- 工具发现机制（硬编码 / 配置文件 / 环境变量）
- Schema 格式（OpenAI 格式 / Anthropic 格式 / 自定义）
- 进程管理方式（同进程 / 子进程 / HTTP）
- 错误处理约定

每增加一种客户端，适配工作量线性增长。这就是 MCP 要解决的问题。

### 2.2 没有统一协议时的问题清单

在 MCP 出现之前，Agent 应用与外部工具的对接面临以下工程问题：

| 问题 | 具体表现 |
|------|---------|
| 重复适配 | N 个客户端 × M 套工具 = N×M 份适配代码 |
| 工具发现无标准 | 每个平台用自己的方式描述和注册工具 |
| Schema 分散 | 同一工具的 Schema 在多个客户端中重复定义且可能不同步 |
| 进程耦合 | 工具通常与 Agent 应用同进程运行，隔离性差 |
| 跨语言复用难 | Python 实现的工具无法被 Node.js 客户端直接使用 |
| 远程接入缺乏约定 | 通过 HTTP 暴露工具时，各家用不同的路由、认证和序列化方式 |

MCP 把这些问题的解决方案统一成一份协议：工具定义交给 Server，工具发现和调用交给 Client-Server 之间的 JSON-RPC 协议[1]。

### 2.3 MCP 具体做什么

```mermaid
flowchart TD
    subgraph L1[" 模型推理层 "]
        A["模型选择工具、生成参数"]
    end
    subgraph L2[" 应用层 "]
        B["Agent Loop / Prompt / 消息管理"]
    end
    subgraph L3[" MCP 连接层 ◀"]
        C["能力发现 / Schema 传递 / 调用协议"]
    end
    subgraph L4[" 外部能力层 "]
        D["工具函数 / 数据源 / API"]
    end
    L1 --> L2 --> L3 --> L4

    classDef host fill:#ff6b6b,stroke:#c92a2a,color:#fff
classDef client fill:#ffd43b,stroke:#e67700,color:#333
classDef server fill:#51cf66,stroke:#2b8a3e,color:#fff
classDef model fill:#cc5de8,stroke:#862e9c,color:#fff
classDef tool fill:#4dabf7,stroke:#1971c2,color:#fff
classDef tool2 fill:#20c997,stroke:#087f5b,color:#fff
classDef tool3 fill:#ff922b,stroke:#d9480f,color:#fff
classDef user fill:#f783ac,stroke:#c2255c,color:#fff
classDef storage fill:#a9e34b,stroke:#5c940d,color:#333
classDef app fill:#74c0fc,stroke:#1864ab,color:#fff
classDef mcp fill:#63e6be,stroke:#0b7285,color:#fff

    class A model
    class B host
    class C server
    class D tool
```

MCP 在这一层做三件事：

1. **定义**：Server 用 JSON Schema 描述自己有哪些 Tool / Resource / Prompt。
2. **发现**：Client 通过 `tools/list` 等标准方法获知 Server 的能力。
3. **调用**：Client 通过 `tools/call` 请求 Server 执行具体 Tool，Server 返回结构化结果。

## 3 核心概念

**六个参与者各司其职。先统一名词，后续不再重复定义。**

| 术语 | 定义 | 在本章的角色 |
|------|------|-------------|
| Host | 运行 Agent 的应用程序 | 创建 Client，管理连接生命周期 |
| Client | 与 Server 维持 1:1 连接的协议实现 | 发送 JSON-RPC 请求，接收响应 |
| Server | 暴露能力并响应调用的进程 | 提供 Tools / Resources / Prompts |
| Tool | 模型可调用、返回结果的函数 | `read_file`、`search_text`、`repo_status` |
| Resource | 暴露给模型的只读数据 | 项目文件内容（URI 寻址） |
| Prompt | Server 预定义的提示模板 | 代码审查模板、提交信息模板 |
| Transport | Client 与 Server 之间的通信方式 | stdio（进程管道）或 HTTP+SSE |
| JSON-RPC | MCP 的消息格式 | 请求 `{method, params}`、响应 `{result}`、错误 `{error}` |

Tool、Resource 和 Prompt 是 MCP 暴露能力的三种原语：

- **Tool**：可带参数、执行动作、返回结果。对应 Function Calling 中的 Tool 概念。
- **Resource**：只读数据，通过 URI 寻址，可订阅变更。Function Calling 中没有直接对应物。
- **Prompt**：Server 维护的可复用 Prompt 模板，Client 发现后注入上下文。

本章实验聚焦 Tools——因为它与第 4 章的 Function Calling 直接对应。Resources 和 Prompts 在 §5.2 和 §5.3 分别说明。

### 3.1 MCP 的术语边界

以下概念**不属于** MCP 规范本身，但常被混淆：

| 概念 | 实际归属 | 本章是否覆盖 |
|------|---------|-------------|
| Function Calling / Tool Use | 模型能力，模型与应用之间的协议 | 第 4 章已讲 |
| Agent Loop | 应用程序控制逻辑 | 第 8 章 |
| 工具函数实现 | 业务代码，放在 Server 中 | 本章实验部分 |
| MCP Server Registry / 市场 | 社区生态，非规范 | 不覆盖 |
| Sampling（Server 请求模型生成） | MCP 草案能力，不稳定 | 不覆盖 |

## 4 完整流程

**一次 MCP 调用分三个阶段：能力发现、能力调用、结果返回。每一步都是标准 JSON-RPC 消息。**

### 4.1 参与者关系

```mermaid
graph TD
    U["用户"] --> H["Host<br/>Agent 应用"]
    H --> C["MCP Client<br/>协议实现"]
    C -->|"JSON-RPC over stdio"| S["MCP Server<br/>独立进程"]
    S --> F1["read_file"]
    S --> F2["search_text"]
    S --> F3["repo_status"]
    F1 --> FS["文件系统"]
    F3 --> G["git"]
    H --> M["大模型"]

    classDef host fill:#ff6b6b,stroke:#c92a2a,color:#fff
classDef client fill:#ffd43b,stroke:#e67700,color:#333
classDef server fill:#51cf66,stroke:#2b8a3e,color:#fff
classDef model fill:#cc5de8,stroke:#862e9c,color:#fff
classDef tool fill:#4dabf7,stroke:#1971c2,color:#fff
classDef tool2 fill:#20c997,stroke:#087f5b,color:#fff
classDef tool3 fill:#ff922b,stroke:#d9480f,color:#fff
classDef user fill:#f783ac,stroke:#c2255c,color:#fff
classDef storage fill:#a9e34b,stroke:#5c940d,color:#333
classDef app fill:#74c0fc,stroke:#1864ab,color:#fff
classDef mcp fill:#63e6be,stroke:#0b7285,color:#fff

    class U user
    class H host
    class C client
    class S server
    class F1 tool
    class F2 tool2
    class F3 tool3
    class FS storage
    class G tool2
    class M model
```

Host 不直接与 Server 通信——Client 是中间的协议桥梁。Server 作为一个独立进程运行，通过 stdio 管道接收 JSON-RPC 消息。模型在 Host 侧，通过 Function Calling 使用 MCP 发现回来的 Tools。

### 4.2 完整时序

```mermaid
sequenceDiagram
    participant U as 用户
    participant H as Host（Agent 应用）
    participant C as MCP Client
    participant S as MCP Server
    participant F as 工具函数
    participant M as 大模型

    U->>H: 查看 README.md
    H->>C: 启动 Client，连接 Server
    C->>S: initialize
    S-->>C: 协议版本、Server 能力
    C->>S: tools/list
    S-->>C: Tool 列表 (read_file, search_text, repo_status)
    C-->>H: 返回 Tools
    H->>M: messages + Tools
    M-->>H: Tool Call: read_file(path='README.md')
    H->>C: tools/call
    C->>S: tools/call {read_file, {path:'README.md'}}
    S->>F: read_file('README.md')
    F-->>S: 文件内容
    S-->>C: Tool Result
    C-->>H: Tool Result
    H->>M: Tool Result
    M-->>H: 文件内容是...
    H-->>U: 最终回答
```

关键观察：

1. **`tools/list` 发生在模型调用之前**——模型需要先知道有哪些工具可用。
2. **Tool Schema 由 Server 维护**——应用不需要手写 Schema。
3. **模型仍然通过 Function Calling 表达调用意图**——MCP 负责的是 Schema 从哪里来、调用怎么送达。
4. **两次模型调用之间穿插了 MCP 往返**——`tools/list` 之后才构建 messages，`tools/call` 之后才回传 Tool Result。

### 4.3 生命周期

```mermaid
stateDiagram-v2
    state "初始化" as INIT
    state "能力协商" as NEGOTIATE
    state "服务中" as SERVING
    state "关闭" as CLOSED

    [*] --> INIT: initialize
    INIT --> NEGOTIATE: 交换协议版本和能力
    NEGOTIATE --> SERVING: 就绪
    SERVING --> SERVING: tools/list
    SERVING --> SERVING: tools/call
    SERVING --> SERVING: resources/read
    SERVING --> CLOSED: 连接断开
    CLOSED --> [*]
```

- **初始化**：Client 发送 `initialize` 请求，Server 返回支持的协议版本和能力列表。
- **服务中**：Client 可以调用 `tools/list`、`tools/call`、`resources/list` 等方法。
- **关闭**：连接断开时结束。stdio transport 的关闭由进程退出触发。

本章实验的生命周期：启动 Server → `initialize` → `tools/list` → `tools/call` → 退出。

### 4.4 传输层：stdio 与 HTTP+SSE

MCP 2025-11-25 稳定版支持两种传输方式：

| 传输方式 | 连接方式 | 适合场景 | 本章使用 |
|---------|---------|---------|---------|
| stdio | 标准输入输出管道 | 本地工具、单机部署 | ✓ |
| Streamable HTTP | HTTP POST + SSE 可选 | 远程工具、跨机器部署 | — |

**stdio** 是 MCP 最简单也最常用的传输方式。Client 启动 Server 作为子进程，通过 `stdin` 发送 JSON-RPC 请求，从 `stdout` 读取响应。

本章实验全部使用 stdio——它足够让我们观察完整的 MCP 调用链路，不需要网络配置。

**为什么从 HTTP+SSE 迁移到 Streamable HTTP？** 2024-11-05 规范中的 HTTP+SSE 传输要求 Server 维护 Session 状态，在大规模部署和负载均衡场景下成本较高。2025-11-25 稳定版引入 Streamable HTTP，在 HTTP POST 基础上选择性使用 SSE，Server 可无状态部署，基础设施兼容性更好[1]。

**即将到来的 2026 RC 进一步简化：** 2026-07-28 Release Candidate 删除了 `initialize`/`initialized` 握手和协议级 Session，核心协议收缩为无状态连接——Client 每次请求直连 Server，不再预先协商能力。Roots、Sampling、Logging 标记弃用，Tasks 和 MCP Apps 进入扩展体系。正式发布窗口在 2026 年下半年，当前生产环境仍应基于 2025-11-25 稳定版[1]。

## 5 原理拆解

**Tools、Resources、Prompts 是 MCP 的三种能力原语。三者的协议结构和适用场景不同，但都通过统一的方法暴露给 Client。**

### 5.1 Tools：让模型可以调用的函数

Tool 是 MCP 最常用的原语。一个 Tool 包含：

```json
{
  "name": "read_file",
  "description": "读取仓库中指定路径的文件内容",
  "inputSchema": {
    "type": "object",
    "properties": {
      "path": {
        "type": "string",
        "description": "文件路径，相对于仓库根目录"
      }
    },
    "required": ["path"]
  }
}
```

这与第 4 章的 Tool Schema 结构一致——MCP Server 返回的 Tool 定义可以直接映射为 Function Calling 的 Tool Schema。

**调用流程：**

1. Client 调用 `tools/list` → Server 返回所有 Tool 定义。
2. 应用把 Tool 定义放入模型请求。
3. 模型返回 Tool Call → 应用通过 `tools/call` 把参数发给 Server。
4. Server 执行函数 → 返回 `{"content": [{"type": "text", "text": "..."}]}`。
5. 应用把结果封装为 Tool Result 交还模型。

### 5.2 Resources：只读数据的标准化暴露

Resource 是 MCP 中与 Tool 互补的原语。

| 特性 | Tool | Resource |
|------|------|----------|
| 读写性质 | 可执行动作、可修改状态 | 只读 |
| 寻址方式 | 按名称调用 | 按 URI 寻址 |
| 参数 | 有（inputSchema） | URI 模板（如 `file:///src/main.py`） |
| 典型用途 | 搜索文件、获取状态、运行命令 | 读取配置、查看日志、获取文件内容 |
| 是否消耗模型 token | Tool Call + Tool Result 都在上下文中 | Resource 内容直接进入上下文 |

Resource 适合暴露 Agent 每次对话可能需要的背景数据——项目配置、构建输出、变更日志等。它们不需要参数，只需要被读取。

本章实验中 `read_file` 仍用 Tool 实现（因为它需要参数 `path`），但生产环境中把静态文件暴露为 Resource 可以减少 Tool Call 往返。

### 5.3 Prompts：可复用的提示模板

Prompt 是 Server 预定义的提示词模板，Client 发现后可以注入到模型上下文中。

```json
// prompts/list 响应示例
{
  "prompts": [
    {
      "name": "code_review",
      "description": "生成代码审查报告",
      "arguments": [
        {"name": "language", "description": "编程语言", "required": true}
      ]
    }
  ]
}
```

与第 3 章 Prompt 模板的区别：

| | 第 3 章 Prompt 模板 | MCP Prompt |
|---|---|---|
| 维护者 | 应用开发者 | Server 开发者 |
| 变更频率 | 跟随应用发布 | 独立于应用 |
| 复用范围 | 单个应用内 | 多个 Client 共享 |
| 典型内容 | 角色、任务、输出协议 | 领域模板（代码审查、提交信息、SQL 审查） |

一个数据库 MCP Server 可以提供 SQL 审查和查询优化 Prompt 模板。所有使用该 Server 的 Agent 客户端都可以通过 `prompts/list` 发现并使用它们，而不需要各自维护一份。

### 5.4 协议消息一览

以下是本章涉及的核心 JSON-RPC 方法速查：

| 方法 | 方向 | 作用 | 何时调用 |
|------|------|------|---------|
| `initialize` | C→S | 交换协议版本和 Server 能力 | 连接建立时 |
| `notifications/initialized` | C→S | 通知 Server 初始化完成 | `initialize` 之后 |
| `tools/list` | C→S | 获取所有可用 Tool 定义 | 构建模型 messages 之前 |
| `tools/call` | C→S | 调用指定 Tool 并获取结果 | 模型返回 Tool Call 之后 |
| `resources/list` | C→S | 获取所有可用 Resource | 需要背景数据时 |
| `resources/read` | C→S | 读取指定 URI 的 Resource | 模型需要具体数据时 |
| `prompts/list` | C→S | 获取所有可用 Prompt | 构建上下文时 |
| `prompts/get` | C→S | 获取指定 Prompt 的完整内容 | 选定后注入 |

本章实验重点观察 `initialize` → `tools/list` → `tools/call` 三个方法。其余方法在 §5.2 和 §5.3 中已结合原语说明。

## 6 渐进实验：代码仓库分析助手

**先用本地 Tool Calling 实现，再用 MCP 实现完全相同的功能。对比的目的不是比性能——而是让你亲眼看到 MCP 在哪里省了代码、在哪里增加了复杂度。**

### 实验目标

实现一个代码仓库分析助手，包含三个工具：

| 工具名 | 功能 | 参数 | 输出 |
|--------|------|------|------|
| `read_file` | 读取文件内容 | `path: string` | 文件文本 |
| `search_text` | 搜索包含关键词的行 | `pattern: string` | 匹配行列表 |
| `repo_status` | 获取仓库当前状态 | 无 | 分支名、最新 commit |

### 三个实验的对照设计

| 实验 A：本地 | 实验 B：MCP stdio | 实验 C：MCP HTTP |
|---|---|---|
| 函数实现 | `tool_functions.py` | `tool_functions.py` | `tool_functions.py` |
| Tool Schema 定义 | 应用代码中硬编码 | MCP Server 通过 `tools/list` 返回 | MCP Server 通过 `tools/list` 返回 |
| 函数调用方式 | Python 直接调用 | JSON-RPC over stdio | JSON-RPC over HTTP |
| 进程边界 | 同进程 | 子进程（stdio 管道） | 子进程（HTTP 端口） |
| 复用给其他客户端 | 复制代码 | 连接同一个 Server | 跨机器共享同一个 Server |

两个实验的查询完全相同：**"查看 README.md 的内容，搜索有没有 TODO"**。

本章没有真实模型调用——全部用固定 Tool Call 和本地执行来观察协议行为。想看 MCP + 真实模型的完整链路，可以把 MCP Client 返回的 Tool Schema 代入第 4 章的 Tool Calling Loop。

### 6.1 实验准备：三个工具函数的纯 Python 实现

`chapters/05-mcp/examples/tool_functions.py` 已经准备好了。下面在 notebook 中验证它们的行为。

这三个函数**不包含任何 MCP 代码**。实验 A 和实验 B 共享同一份实现。

In [1]:
import os
import sys
from pathlib import Path

# 检测仓库根目录：从当前工作目录向上查找 chapters/05-mcp/
def _find_repo_root():
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "chapters" / "05-mcp").is_dir():
            return start
    # 回退：假设 cwd 就是仓库根
    return Path.cwd()

REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "chapters" / "05-mcp" / "examples"))

print(f"Repo root: {REPO_ROOT}")

from tool_functions import read_file, search_text, repo_status

# 验证 read_file
content = read_file("README.md")
print("=== read_file('README.md') ===")
print(content[:200])
assert "Demo Repo" in content

# 验证 search_text
matches = search_text("TODO")
print("\n=== search_text('TODO') ===")
for m in matches:
    print(f"  {m['file']}:{m['line']}: {m['content']}")
assert len(matches) >= 1

# 验证 repo_status
status = repo_status()
print("\n=== repo_status() ===")
print(f"  branch: {status['branch']}")
print(f"  commit: {status['latest_commit'][:50]}")
assert "branch" in status
assert "latest_commit" in status

print("\n三个函数均可独立运行。它们不依赖任何 MCP 代码。")

Repo root: /Users/dymo/Develop/notes/docs/ai/agent
=== read_file('README.md') ===
# Demo Repo

This is a minimal sample repository used by Chapter 5's code-repo-analyzer experiments.

It contains a few small source files so that `read_file`, `search_text`, and `repo_status` have so

=== search_text('TODO') ===
  src/main.py:5: """Return a greeting string.  TODO: add i18n support."""

=== repo_status() ===
  branch: unknown
  commit: git not available

三个函数均可独立运行。它们不依赖任何 MCP 代码。


fatal: your current branch 'master' does not have any commits yet


### 6.2 实验 A：本地 Tool Calling

下面用第 4 章的模式实现本地 Tool Calling。关键点：

- Tool Schema 硬编码在应用代码中（三个 Python dict）。
- 函数调用在同一进程中直接执行。
- 用固定 Tool Call 模拟模型返回，不调真实 API。

In [2]:
import json


# ── 三份 Tool Schema，直接定义在应用代码中 ──

LOCAL_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "读取仓库中指定路径的文件内容。path 相对于仓库根目录。",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "文件路径"}
                },
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_text",
            "description": "搜索仓库所有文件中包含关键词的文本行。",
            "parameters": {
                "type": "object",
                "properties": {
                    "pattern": {"type": "string", "description": "搜索关键词"}
                },
                "required": ["pattern"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "repo_status",
            "description": "获取仓库当前状态：分支名和最新 commit。",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False,
            },
        },
    },
]

print(f"本地定义了 {len(LOCAL_TOOLS)} 个 Tool Schema：")
for t in LOCAL_TOOLS:
    print(f"  - {t['function']['name']}")
assert len(LOCAL_TOOLS) == 3

本地定义了 3 个 Tool Schema：
  - read_file
  - search_text
  - repo_status


In [3]:
# ── 模拟模型返回 Tool Call ──
# 用户输入："查看 README.md 的内容，搜索有没有 TODO"

fixed_tool_calls = [
    {
        "id": "call_local_001",
        "type": "function",
        "function": {
            "name": "read_file",
            "arguments": json.dumps({"path": "README.md"}),
        },
    },
    {
        "id": "call_local_002",
        "type": "function",
        "function": {
            "name": "search_text",
            "arguments": json.dumps({"pattern": "TODO"}),
        },
    },
]

print("模型返回的 Tool Calls：")
for tc in fixed_tool_calls:
    f = tc["function"]
    print(f"  {tc['id']}: {f['name']}({f['arguments']})")
assert len(fixed_tool_calls) == 2
assert fixed_tool_calls[0]["function"]["name"] == "read_file"
assert fixed_tool_calls[1]["function"]["name"] == "search_text"


模型返回的 Tool Calls：
  call_local_001: read_file({"path": "README.md"})
  call_local_002: search_text({"pattern": "TODO"})


In [4]:
# ── 根据名称分发 Tool Call 到对应函数 ──

TOOL_DISPATCHER = {
    "read_file": lambda args: read_file(args["path"]),
    "search_text": lambda args: search_text(args["pattern"]),
    "repo_status": lambda args: repo_status(),
}


def execute_tool_calls(tool_calls: list[dict]) -> list[dict]:
    # 执行 Tool Calls 列表，返回 Tool Results。
    results = []
    for tc in tool_calls:
        func_name = tc["function"]["name"]
        arguments = json.loads(tc["function"]["arguments"])
        func = TOOL_DISPATCHER[func_name]
        try:
            output = func(arguments)
            results.append({
                "tool_call_id": tc["id"],
                "role": "tool",
                "content": json.dumps(output, ensure_ascii=False),
            })
            print(f"OK {tc['id']} ({func_name}): success")
        except Exception as exc:
            results.append({
                "tool_call_id": tc["id"],
                "role": "tool",
                "content": json.dumps({"error": str(exc)}),
            })
            print(f"ERR {tc['id']} ({func_name}): {exc}")
    return results


tool_results = execute_tool_calls(fixed_tool_calls)

print("\nTool Results：")
for tr in tool_results:
    preview = tr["content"][:120]
    print(f"  {tr['tool_call_id']}: {preview}...")

assert len(tool_results) == 2
assert tool_results[0]["tool_call_id"] == "call_local_001"
assert "Demo Repo" in tool_results[0]["content"]
assert "TODO" in tool_results[1]["content"]


OK call_local_001 (read_file): success
OK call_local_002 (search_text): success

Tool Results：
  call_local_001: "# Demo Repo\n\nThis is a minimal sample repository used by Chapter 5's code-repo-analyzer experiments.\n\nIt contains a...
  call_local_002: [{"file": "src/main.py", "line": 5, "content": "\"\"\"Return a greeting string.  TODO: add i18n support.\"\"\""}]...


### 本地 Tool Calling 小结

- **Tool Schema** 由应用开发者手写，放在应用代码里。
- **工具发现** 不存在——工具名直接硬编码在 `TOOL_DISPATCHER` 字典中。
- **函数调用** 在同进程中直接执行，没有协议开销。
- **如果另一个 Agent 需要这套工具**，需要复制 Tool Schema 和函数实现。

现在用 MCP 重做一次，观察差异。

### 6.3 实验 B：通过 MCP 连接同一套工具

**MCP Server 代码**详见 `chapters/05-mcp/examples/mcp_server.py`。

以下是在 notebook 中启动 MCP Client 并观察完整调用链路的代码。

先确认 `mcp` 包已安装：

In [5]:
try:
    import mcp
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
    ver = getattr(mcp, "__version__", "unknown")
    print(f"mcp version: {ver}")
except ImportError:
    print("mcp not installed. Run: uv add mcp")
    raise

mcp version: unknown


In [6]:
import asyncio
import json
from pathlib import Path

# Server 脚本路径
SERVER_SCRIPT = str(
    Path.cwd() / "chapters" / "05-mcp" / "examples" / "mcp_server.py"
)


async def discover_tools():
    # 启动 MCP Server，通过 tools/list 发现可用工具。
    server_params = StdioServerParameters(
        command="uv",
        args=["run", "python", SERVER_SCRIPT],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # 步骤 1：初始化
            init_result = await session.initialize()
            print("=== initialize ===")
            print(f"  protocol version: {init_result.protocolVersion}")
            print(f"  server name: {init_result.serverInfo.name}")

            # 步骤 2：获取工具列表
            tools_result = await session.list_tools()
            print(f"\n=== tools/list ===")
            print(f"  found {len(tools_result.tools)} tools:")
            for tool in tools_result.tools:
                print(f"    - {tool.name}: {tool.description}")

            return init_result, tools_result


init_result, tools_result = await discover_tools()

# 验证工具数量
assert len(tools_result.tools) == 3
tool_names = {t.name for t in tools_result.tools}
assert tool_names == {"read_file", "search_text", "repo_status"}
print("\nOK: discovered the same 3 tools as Experiment A via MCP protocol")

=== initialize ===
  protocol version: 2025-11-25
  server name: repo-analyzer

=== tools/list ===
  found 3 tools:
    - read_file: 读取仓库中指定路径的文件内容。path 相对于仓库根目录。
    - search_text: 搜索仓库所有文件中包含关键词的文本行。返回匹配列表。
    - repo_status: 获取仓库当前状态：分支名和最新 commit。

OK: discovered the same 3 tools as Experiment A via MCP protocol


In [7]:
async def call_tools_in_session(tool_calls: list[tuple]) -> list[dict]:
    # 在一个 ClientSession 内执行多次 tools/call。
    # 只启动一次 Server 进程、一次 initialize，展示 MCP 的正确用法。
    server_params = StdioServerParameters(
        command="uv",
        args=["run", "python", SERVER_SCRIPT],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            results = []
            for tool_name, arguments in tool_calls:
                # tools/call
                result = await session.call_tool(tool_name, arguments)

                # MCP Tool 返回的是 ContentBlock 列表
                contents = []
                for block in result.content:
                    if hasattr(block, "text"):
                        contents.append(block.text)

                results.append({
                    "tool_name": tool_name,
                    "arguments": arguments,
                    "content": "\n".join(contents),
                })

            return results


# 在一个 Session 中执行与实验 A 完全相同的两个操作。
# 只启动一次 Server 进程——这才是 MCP 的正确用法。
results = await call_tools_in_session([
    ("read_file", {"path": "README.md"}),
    ("search_text", {"pattern": "TODO"}),
])

for r in results:
    print(f"=== {r['tool_name']}({json.dumps(r['arguments'])}) ===")
    print(r["content"][:200])
    print()

assert "Demo Repo" in results[0]["content"], f"Unexpected content: {results[0]['content'][:200]}"
assert "TODO" in results[1]["content"], f"Unexpected content: {results[1]['content'][:200]}"
print("OK: MCP results match Experiment A exactly (single session, 1 initialize, 2 calls)")

=== read_file({"path": "README.md"}) ===
# Demo Repo

This is a minimal sample repository used by Chapter 5's code-repo-analyzer experiments.

It contains a few small source files so that `read_file`, `search_text`, and `repo_status` have so

=== search_text({"pattern": "TODO"}) ===
src/main.py:5: """Return a greeting string.  TODO: add i18n support."""

OK: MCP results match Experiment A exactly (single session, 1 initialize, 2 calls)


In [8]:
# ── 对比：Schema 定义方式 ──

print("=== Experiment A: Local Tool Schema ===")
print("Location: hardcoded as Python dict in application code")
print(f"Tool count: {len(LOCAL_TOOLS)}")
print()

print("=== Experiment B: MCP Tool Schema ===")
print("Location: in MCP Server process, retrieved via tools/list protocol")
print(f"Tool count: {len(tools_result.tools)}")
print()

# 对比 Schema 内容
mcp_schemas = []
for tool in tools_result.tools:
    mcp_schemas.append({
        "name": tool.name,
        "description": tool.description,
        "input_schema_keys": (
            list(tool.inputSchema.get("properties", {}).keys())
            if tool.inputSchema else []
        ),
    })

for ms in mcp_schemas:
    print(f"  {ms['name']}: params={ms['input_schema_keys']}")

print()
print("MCP Server defines Schema once; multiple Clients retrieve it via protocol.")
print("Experiment A: change read_file params -> update every codebase manually.")
print("Experiment B: change mcp_server.py -> all Clients auto-fetch via tools/list.")

=== Experiment A: Local Tool Schema ===
Location: hardcoded as Python dict in application code
Tool count: 3

=== Experiment B: MCP Tool Schema ===
Location: in MCP Server process, retrieved via tools/list protocol
Tool count: 3

  read_file: params=['path']
  search_text: params=['pattern']
  repo_status: params=[]

MCP Server defines Schema once; multiple Clients retrieve it via protocol.
Experiment A: change read_file params -> update every codebase manually.
Experiment B: change mcp_server.py -> all Clients auto-fetch via tools/list.


### 6.4 对比总结

| 维度 | 实验 A：本地 Tool Calling | 实验 B：MCP |
|------|--------------------------|------------|
| 工具函数实现 | `tool_functions.py` | `tool_functions.py`（同一份） |
| Tool Schema 位置 | 应用代码硬编码 | Server 进程中，通过 `tools/list` 获取 |
| 发现机制 | 无——名称硬编码在 Dispatcher 中 | `tools/list` 协议，运行时发现 |
| 函数调用 | 同进程直接调用 | 跨进程（子进程 stdio 管道） |
| 复用给新客户端 | 复制代码 | 连接同一个 Server |
| 额外依赖 | 无 | `mcp` 包 + 子进程管理 |
| 错误模式 | Python 异常直接捕获 | 需要区分 JSON-RPC 错误和函数执行错误 |
| 适用工具数 | 少量（< 10 个）适用 | 不限 |

**结论：3 个工具、1 个客户端的场景下，MCP 的额外复杂度超过收益。** 当工具数增长、多个客户端需要对接同一套外部能力时，标准协议的复用价值才开始超过协议本身的开销。

这个结论很重要——它回答了「为什么不是所有项目都应该用 MCP」。

### 6.5 实验 C：远程 MCP（Streamable HTTP）

实验 B 使用 stdio 传输。stdio 适合本地工具，但跨机器场景需要 HTTP。下面启动同一个 Server 的 HTTP 版本，通过 `streamablehttp_client` 远程连接，执行相同的工具调用。

Server 代码见 `chapters/05-mcp/examples/mcp_server_http.py`。与实验 B 的唯一区别：`mcp.run(transport="streamable-http")` 替代了 `mcp.run(transport="stdio")`。工具函数完全复用 `tool_functions.py`。

In [9]:
import subprocess
import time
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

SERVER_SCRIPT_HTTP = str(
    Path.cwd() / "chapters" / "05-mcp" / "examples" / "mcp_server_http.py"
)
SERVER_URL = "http://localhost:8000/mcp"

# 1. 启动 HTTP Server 子进程
server_proc = subprocess.Popen(
    ["uv", "run", "python", SERVER_SCRIPT_HTTP],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print(f"Server PID: {server_proc.pid}")

# 2. 轮询直到 Server 就绪（尝试向 MCP endpoint 发 POST）
ready = False
for _ in range(40):
    time.sleep(0.15)
    if server_proc.poll() is not None:
        raise RuntimeError(f"Server exited with code {server_proc.returncode}")
    try:
        # MCP streamable-http endpoint 接受 POST 请求
        resp = httpx.post(SERVER_URL, timeout=0.5)
        ready = True
        break
    except (httpx.ConnectError, httpx.ReadError, httpx.RemoteProtocolError):
        pass

assert ready, "Server did not become ready in 6 seconds"
print("Server ready")


async def remote_experiment():
    async with streamablehttp_client(SERVER_URL) as (read, write, _get_session_id):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 3. 发现工具
            tools_result = await session.list_tools()
            print(f"tools/list: {len(tools_result.tools)} tools")
            for t in tools_result.tools:
                print(f"  - {t.name}")

            # 4. 调用工具
            r1 = await session.call_tool("read_file", {"path": "README.md"})
            r2 = await session.call_tool("search_text", {"pattern": "TODO"})

            return tools_result, r1, r2


tools_result, r1, r2 = await remote_experiment()

# 5. 展示结果
readme_text = "".join(b.text for b in r1.content if hasattr(b, "text"))
print(f"\n=== read_file('README.md') via HTTP ===")
print(readme_text[:200])
print(f"\n=== search_text('TODO') via HTTP ===")
for b in r2.content:
    if hasattr(b, "text"):
        print(b.text[:200])

assert "Demo Repo" in readme_text
assert len(tools_result.tools) == 3
print("\nOK: remote MCP results match stdio results exactly")

# 6. 关闭 Server
server_proc.terminate()
server_proc.wait()
print("Server stopped")

Server PID: 48433


Server ready
tools/list: 3 tools
  - read_file
  - search_text
  - repo_status

=== read_file('README.md') via HTTP ===
# Demo Repo

This is a minimal sample repository used by Chapter 5's code-repo-analyzer experiments.

It contains a few small source files so that `read_file`, `search_text`, and `repo_status` have so

=== search_text('TODO') via HTTP ===
src/main.py:5: """Return a greeting string.  TODO: add i18n support."""

OK: remote MCP results match stdio results exactly


Server stopped


### 三实验对比

| 维度 | 实验 A：本地 | 实验 B：MCP stdio | 实验 C：MCP HTTP |
|------|------------|-------------------|-------------------|
| 工具函数 | `tool_functions.py` | `tool_functions.py` | `tool_functions.py` |
| Schema 来源 | 应用硬编码 | `tools/list` (stdio) | `tools/list` (HTTP) |
| 传输方式 | 同进程调用 | 子进程 stdio 管道 | 子进程 HTTP |
| 跨机器 | 否 | 否 | 是 |
| 额外依赖 | 无 | `mcp` | `mcp` + HTTP 端口管理 |
| 适合场景 | 单客户端、少量工具 | 多工具、本地复用 | 多客户端、跨机器共享 |

实验 B 和 C 使用同一套函数和同一套 MCP 协议。唯一的区别是 `mcp.run(transport=...)` 的参数和 Client 的连接方式。这就是 MCP 的核心价值：传输层可以切换，工具定义和调用协议不变。

## 7 工程边界

**本章实验展示了 MCP 的基本原理。生产环境还需要处理以下问题。**

### 7.1 传输层选择

| 场景 | 推荐传输 | 原因 |
|------|---------|------|
| 本地工具（IDE 插件、CLI Agent） | stdio | 零网络配置，进程即隔离 |
| 远程工具（团队共享的数据库工具、CI 状态服务） | HTTP+SSE | 可跨机器，可用已有鉴权基础设施 |
| 容器化部署 | HTTP+SSE | 容器间通信不依赖进程父子关系 |

stdio 的局限：不适合需要从多个 Host 同时连接的场景。HTTP+SSE 需要额外处理服务发现和健康检查。

### 7.2 Server 发现与生命周期

MCP 规范没有规定「如何找到 MCP Server」。这是规范刻意留下的空白——不同平台有不同的配置方式：

- Claude Desktop：通过 `claude_desktop_config.json` 指定 Server 路径。
- VS Code Copilot：通过扩展注册。
- 自建 Agent：应用代码中硬编码或配置文件指定。

Server 的生命周期管理（启动、重启、健康检查、优雅关闭）也由 Host 负责。规范只定义了连接建立后的协议[1]。

### 7.3 错误处理

MCP 有两层错误需要区分：

| 错误层 | 来源 | 示例 | 处理方式 |
|--------|------|------|---------|
| JSON-RPC 错误 | Server 无法处理请求 | `Method not found`、`Invalid params` | 检查方法名和参数格式 |
| 工具执行错误 | 函数内部异常 | 文件不存在、权限不足、路径穿越 | 在 Server 端捕获并以 `isError: true` 返回 |
| 传输错误 | 连接断开、进程崩溃 | Server 意外退出、管道损坏 | 重连 + 指数退避 |

MCP 的 `tools/call` 响应包含 `isError` 字段。Server 端应该在函数内部捕获异常并通过 `isError: true` 返回错误信息——而不是让 JSON-RPC 层直接报错。

### 7.4 权限与信任

Server 暴露的能力可能涉及敏感操作：

- `read_file` 需要限制到仓库根目录以内（路径穿越防护）。
- `repo_status` 可能暴露 commit 作者和消息。
- 未来增加了 `run_command` 工具后，命令注入风险。

MCP 规范提供了权限框架的基础，但具体策略由 Host 实现：

1. **工具白名单**：Host 只向模型暴露经过筛选的工具子集。
2. **参数校验**：路径、命令等参数在 Server 侧做二次校验。
3. **用户确认**：在调用写操作前要求用户确认。

本章 `tool_functions.py` 中的 `read_file` 已经在函数层面做了路径穿越防护。生产环境中这一层校验应该在 Server 的每个 Tool 入口处统一执行。

### 7.5 工具结果过大

文件内容可能超出模型上下文窗口。三种处理策略：

| 策略 | 适用场景 | 本章实验 |
|------|---------|---------|
| 不做截断 | 已知文件很小 | ✓——`README.md` 和 `main.py` 只有几行 |
| Server 端截断 | 大文件读取、日志查询 | 未实现 |
| Client 端限制 | 控制上下文 token 预算 | 未实现 |

生产环境中建议 Server 端设置合理的默认截断上限（如 8000 字符），并支持通过参数显式请求更多内容。

### 7.6 常见误区

| 误区 | 事实 |
|------|------|
| "MCP 是 Function Calling 的替代品" | MCP 和 Function Calling 在不同层次。前者连接应用与工具，后者连接模型与应用。 |
| "用了 MCP 就不需要管安全了" | MCP 的权限模型是基础框架，文件系统权限、路径校验、命令注入防护仍然是应用层责任。 |
| "所有工具都应该走 MCP" | 少量工具 + 单客户端场景下，本地 Tool Calling 更简单直接。MCP 解决的是多客户端复用的协调成本。 |
| "MCP Server 可以无脑接入任何 Agent" | Agent 仍然需要理解 Tool 的语义。协议解决发现和调用格式，但 prompt 设计和 Tool Choice 逻辑仍由应用控制。 |
| "MCP 比本地调用更快" | MCP 增加了序列化、进程启动和网络往返，"更快" 不是它的目标。它的价值在工程复用。 |

## 8 MCP 与 Function Calling 的关系

**两者不在同一层，不存在替代关系。**

| 维度 | Function / Tool Calling | MCP |
|------|------------------------|-----|
| 所处层次 | 模型 ↔ 应用 | 应用 ↔ 能力提供方 |
| 主要职责 | 模型表达调用意图和参数，应用执行并回传结果 | 能力发现、Schema 描述、连接与调用协议 |
| 是否依赖模型 | 是——没有模型就没有 Tool Call | 否——协议本身与模型无关 |
| Schema 维护方 | 应用开发者 | Server 开发者 |
| 跨客户端复用 | 每个客户端单独定义 Schema | 通过统一协议，一次定义 |
| 简单进程内工具 | 足够，不建议引入 MCP | 引入 MCP 增加进程管理和协议开销 |
| 与第 4 章的关系 | 第 4 章已讲透 | 本章实验 B 中的 Tool Schema 来自 `tools/list`，替代了第 4 章中手写的 Schema |

一句话总结：

> **Function Calling 是模型表达"我要调用什么"的协议。MCP 是应用找到"有什么可以调用"并用统一方式调用的协议。两者协作：MCP 负责发现和传输，Function Calling 负责让模型理解和选择。**

### 8.1 两者怎么一起工作

```mermaid
flowchart TD
    S["MCP Server"] -->|"tools/list"| T["Tool 定义<br/>(read_file, search_text, repo_status)"]
    T --> A1["应用: 转为 Function Calling<br/>Tool Schema → 发给模型"]
    A1 --> M1["模型: 返回 Tool Call<br/>决定调用哪个工具、什么参数"]
    M1 --> A2["应用: 通过 tools/call<br/>发给 MCP Server"]
    A2 --> R["获得 Tool Result"]
    R --> A3["应用: 封装为 Tool Result<br/>回传给模型"]
    A3 --> M2["模型: 根据结果<br/>生成最终回答"]

    classDef host fill:#ff6b6b,stroke:#c92a2a,color:#fff
classDef client fill:#ffd43b,stroke:#e67700,color:#333
classDef server fill:#51cf66,stroke:#2b8a3e,color:#fff
classDef model fill:#cc5de8,stroke:#862e9c,color:#fff
classDef tool fill:#4dabf7,stroke:#1971c2,color:#fff
classDef tool2 fill:#20c997,stroke:#087f5b,color:#fff
classDef tool3 fill:#ff922b,stroke:#d9480f,color:#fff
classDef user fill:#f783ac,stroke:#c2255c,color:#fff
classDef storage fill:#a9e34b,stroke:#5c940d,color:#333
classDef app fill:#74c0fc,stroke:#1864ab,color:#fff
classDef mcp fill:#63e6be,stroke:#0b7285,color:#fff

    class S server
    class T tool
    class R tool2
    class A1,A2,A3 app
    class M1,M2 model
```

MCP 负责「有什么工具」和「怎么调用」，Function Calling 负责「该调用哪个」和「参数是什么」。两者配合时，存在两种主要架构：

**架构 A：Host 自行桥接。** 本章实验和大多数自建 Agent 采用此模式。MCP 的 `tools/list` 产出 Schema，应用将其转换为 Function Calling 的 `tools` 参数。模型返回 Tool Call 后，应用通过 MCP Client 的 `tools/call` 执行。这是最灵活的方式，应用完全控制工具发现、参数校验和权限。

**架构 B：平台托管 MCP。** OpenAI Responses API 等平台已原生支持 MCP 连接器——平台直接调用 `mcp_list_tools` 发现工具并注入模型上下文，模型返回的 Tool Call 也由平台直接通过 MCP 执行。应用只需要提供 MCP Server 的地址和凭证，不参与中间的协议转换。

架构 A 适合需要完整控制的应用场景；架构 B 适合希望减少胶水代码、快速接入的平台型产品。两种架构都使用 MCP 作为连接协议，区别在于桥接代码的归属。

## 9 MCP 的热度与实际采用

**MCP 解决了真实问题，但它的收益需要一定规模才能显现。热度和实际工程价值之间存在合理差距。**

### 9.1 初期热度来自哪里

MCP 在 2024 年底由 Anthropic 发布后获得大量关注，热度主要来自：

1. **叙事契合**：AI Agent 需要标准化基础设施，MCP 恰好填补了"Agent 到工具"这一层的协议空白。
2. **简单 Server 容易上手**：一个文件就能写 MCP Server，教程级体验好。
3. **Claude Desktop 原生支持**：开箱可用，降低了第一印象的门槛。

但热度和工程验证是两个阶段。一个 Server 的 demo 跑通只需要 30 分钟；让它稳定服务生产流量需要处理错误、权限、版本、监控和合规——这些才是工程工作的主体。

### 9.2 简单场景为什么收益有限

回顾 §6.4 的对比表。3 个工具、1 个客户端的场景下：

| MCP 增加的成本 | MCP 节省的成本 |
|---------------|---------------|
| 学习 `mcp` 包 API | — |
| 管理子进程生命周期 | — |
| 处理 JSON-RPC 错误层 | — |
| 调试跨进程通信 | — |

当只有 1 个客户端和 3 个工具时，这些成本没有对应收益。**MCP 的价值不是线性的——它在多客户端、多工具、跨团队场景下才从"成本"转为"收益"。**

### 9.3 协议核心的主动收缩

除了规模门槛之外，MCP 规范自身也在经历明显的收缩调整。理解这一趋势比记住具体版本号更重要。

2024 年底发布时，MCP 的协议范围较宽：Roots（Client 向 Server 暴露资源）、Sampling（Server 请求模型生成）、Logging 等都在核心规范中。2026 年 Release Candidate 发生了三个方向性变化（以下基于 RC 版本的趋势判断，最终以正式发布为准）：

1. **核心收窄**：`initialize`/`initialized` 握手和协议级 Session 被删除，核心协议退化为无状态——Client 每次请求直连 Server，不再预先协商能力。Roots、Sampling、Logging 标记弃用。
2. **能力外移**：Tasks（长时间运行任务）和 MCP Apps（打包分发）进入扩展体系，不再属于 Core。
3. **治理下沉**：鉴权、速率限制、多租户隔离等由传输层或平台处理，协议本身只定义互操作格式。

这些变化不是衰退——它们是协议从"大而全的早期版本"走向"收窄核心 + 开放扩展"的标志。TCP/IP 和 HTTP 都经历过类似的瘦身周期：当协议成为基础设施后，核心必须足够小才能稳定，扩展性通过外围生态解决。

官方路线仍在持续投入：Streamable HTTP 传输标准化、企业级鉴权（OAuth 2.0 完整流程）、治理框架和生产部署指南都在推进中。因此不能把协议收缩写成 MCP 衰退——它更像 HTTP/1.1 → HTTP/2 → HTTP/3 的演进：每一版都在简化连接模型，同时扩大生态覆盖。

### 9.4 MCP Server 的质量和维护

### 9.5 什么时候应该用 MCP

使用以下检查清单判断：

- [ ] 同一套工具有 ≥2 个 Agent 客户端需要复用？
- [ ] 工具数量 ≥5 个，且可能继续增长？
- [ ] 工具的维护者和 Agent 应用的维护者是不同团队？
- [ ] 工具需要独立于 Agent 应用升级和部署？
- [ ] 希望用标准协议让未来的新客户端"即插即用"？

**以下为课程作者的启发式判断，实际决策需要结合团队规模、现有基础设施和维护预算：满足其中 2 条或以上时，MCP 的收益通常超过成本。**

反之，如果你正在写一个只有 3 个本地工具的个人 Agent 项目——直接导入函数、手写 Schema、用第 4 章的 Tool Calling Loop 就行。等工具数量和客户端数量增长时再迁移到 MCP，协议本身不会因为晚了几个月就失效。

## 10 本章小结

你现在应该能够解释：

1. MCP 为什么出现——解决多客户端重复适配外部能力的工程问题。
2. Host、Client、Server、Tool、Resource、Prompt 六个参与者的职责。
3. `initialize` → `tools/list` → `tools/call` 的完整调用顺序。
4. Tools、Resources、Prompts 三种原语的区别和适用场景。
5. 为什么 MCP 和 Function Calling 不在同一层——前者是应用与工具之间的协议，后者是模型与应用之间的协议。
6. 什么场景该用 MCP（多客户端、多工具、跨团队），什么场景该用本地 Tool Calling（简单、快速、够用）。
7. 生产环境中 MCP 还需要处理的工程问题：传输层选择、Server 生命周期、错误处理、权限和结果截断。

## 11 与产研协作助手的关系

第 11 章的产研协作助手需要连接多个外部能力：代码仓库（读取需求文档和技术方案）、任务系统（查询任务状态）、CI/CD（获取构建和测试结果）。

MCP 可以让这些能力通过统一协议接入：每个外部系统对应一个 MCP Server，产研协作助手作为 Host 通过 Client 发现和调用它们。当 Agent 需要新增一个外部能力时，只需连接新的 Server，不需要修改 Agent 核心代码。

具体接入方案在第 11 章展开。第 6 章先解决另一个问题：产研协作助手如何读取和理解大量的项目文档——这就是 RAG。

## 参考文献

[1] Model Context Protocol. Specification (2025-11-25)[EB/OL]. https://modelcontextprotocol.io/specification, 2026-07-12.

[2] Anthropic. Model Context Protocol: Open standard for connecting AI assistants to tools and data[EB/OL]. https://www.anthropic.com/news/model-context-protocol, 2024-11-25.

[3] Model Context Protocol. Python SDK[EB/OL]. https://github.com/modelcontextprotocol/python-sdk, 2026-07-12.

[4] OpenAI. Function Calling Guide[EB/OL]. https://platform.openai.com/docs/guides/function-calling, 2026-07-12.

[5] OpenAPI Initiative. JSON Schema: A Media Type for Describing JSON Documents (Draft 2020-12)[EB/OL]. https://json-schema.org/draft/2020-12, 2026-07-12.